# CUDA C: tiled matmul, events, sanitizer and profilers

This notebook goes with chapters 5.2 (tiled matmul), 8.3 (fast matmul) and 9.1 (debugging and profiling).
It is all CUDA C++. You will write one .cu file with a naive and a tiled kernel, time both with CUDA events, check
the answer against the CPU, then run it under compute-sanitizer, Nsight Systems and Nsight Compute.

It needs a real NVIDIA GPU with the CUDA toolkit, for example Google Colab with a T4 GPU.

In [ ]:
!nvcc --version
!nvidia-smi --query-gpu=name,compute_cap --format=csv

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>!nvcc --version</code> | Shows the version of the CUDA compiler. |
| 2 | <code>!nvidia-smi --query-gpu=name,compute_cap --format=csv</code> | Shows the GPU name and its compute capability, which we pass to -arch below (7.5 means sm_75). |

## 1. The program

The file has a CUDA_CHECK macro, the two kernels, a CPU reference, and a main that times each kernel with events.

In [ ]:
%%writefile matmul.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e = (call); if (e != cudaSuccess) { \
    printf("CUDA error %s at line %d\n", cudaGetErrorString(e), __LINE__); exit(1); } } while (0)

constexpr int TILE = 16;

__global__ void mmNaive(const float* A, const float* B, float* C, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < n) {
        float acc = 0.0f;
        for (int k = 0; k < n; k++) acc += A[row * n + k] * B[k * n + col];
        C[row * n + col] = acc;
    }
}

__global__ void mmTiled(const float* A, const float* B, float* C, int n) {
    __shared__ float sA[TILE][TILE];
    __shared__ float sB[TILE][TILE];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty, col = blockIdx.x * TILE + tx;
    float acc = 0.0f;
    for (int p = 0; p < (n + TILE - 1) / TILE; p++) {
        int ka = p * TILE + tx, kb = p * TILE + ty;
        sA[ty][tx] = (row < n && ka < n) ? A[row * n + ka] : 0.0f;
        sB[ty][tx] = (kb < n && col < n) ? B[kb * n + col] : 0.0f;
        __syncthreads();
        for (int k = 0; k < TILE; k++) acc += sA[ty][k] * sB[k][tx];
        __syncthreads();
    }
    if (row < n && col < n) C[row * n + col] = acc;
}

void mmCPU(const float* A, const float* B, float* C, int n) {
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++) {
            double acc = 0.0;
            for (int k = 0; k < n; k++) acc += (double)A[i * n + k] * B[k * n + j];
            C[i * n + j] = (float)acc;
        }
}

int main(int argc, char** argv) {
    int n = argc > 1 ? atoi(argv[1]) : 1024;
    size_t bytes = (size_t)n * n * sizeof(float);
    float *A = (float*)malloc(bytes), *B = (float*)malloc(bytes), *C = (float*)malloc(bytes), *R = (float*)malloc(bytes);
    for (int i = 0; i < n * n; i++) { A[i] = (rand() % 100) / 100.0f; B[i] = (rand() % 100) / 100.0f; }
    mmCPU(A, B, R, n);

    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, bytes));
    CUDA_CHECK(cudaMalloc(&dB, bytes));
    CUDA_CHECK(cudaMalloc(&dC, bytes));
    CUDA_CHECK(cudaMemcpy(dA, A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, B, bytes, cudaMemcpyHostToDevice));

    dim3 block(TILE, TILE), grid((n + TILE - 1) / TILE, (n + TILE - 1) / TILE);
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    for (int v = 0; v < 2; v++) {
        CUDA_CHECK(cudaMemset(dC, 0, bytes));
        CUDA_CHECK(cudaEventRecord(start));
        if (v == 0) mmNaive<<<grid, block>>>(dA, dB, dC, n);
        else mmTiled<<<grid, block>>>(dA, dB, dC, n);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        CUDA_CHECK(cudaMemcpy(C, dC, bytes, cudaMemcpyDeviceToHost));
        double maxErr = 0.0;
        for (int i = 0; i < n * n; i++) maxErr = fmax(maxErr, fabs(C[i] - R[i]));
        double gflops = 2.0 * n * n * n / (ms * 1e6);
        printf("%-6s n=%d  %.3f ms  %.1f GFLOP/s  max error %.2e\n", v == 0 ? "naive" : "tiled", n, ms, gflops, maxErr);
    }

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    free(A); free(B); free(C); free(R);
    return 0;
}

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>%%writefile matmul.cu</code> | Save the rest of the cell as matmul.cu. |
| 2 | <code>#include &lt;cstdio&gt;</code> | printf. |
| 3 | <code>#include &lt;cstdlib&gt;</code> | malloc, free, rand, atoi. |
| 4 | <code>#include &lt;cmath&gt;</code> | fabs and fmax for the error check. |
| 5 | <code>#include &lt;cuda_runtime.h&gt;</code> | The CUDA runtime API. |
| 7 | <code>#define CUDA_CHECK(call) do { cudaError_t e = (call); if (e != cudaSuccess) { \</code> | If a CUDA call fails, print the error name and the line, and stop. |
| 8 | <code>printf(&quot;CUDA error %s at line %d\n&quot;, cudaGetErrorString(e), __LINE__); exit(1); } } while (0)</code> | Second half of the macro. do { } while (0) makes it one statement. |
| 10 | <code>constexpr int TILE = 16;</code> | Tile width, known at compile time so the shared arrays can have a fixed size. |
| 12 | <code>__global__ void mmNaive(const float* A, const float* B, float* C, int n) {</code> | The naive kernel. const marks inputs that are only read. |
| 13 | <code>int row = blockIdx.y * blockDim.y + threadIdx.y;</code> | Row of the output this thread computes, from the y part of the index. |
| 14 | <code>int col = blockIdx.x * blockDim.x + threadIdx.x;</code> | Column, from the x part. Consecutive threads get consecutive columns, so B and C accesses coalesce. |
| 15 | <code>if (row &lt; n &amp;&amp; col &lt; n) {</code> | Skip threads outside the matrix. |
| 16 | <code>float acc = 0.0f;</code> | Accumulator in a register. |
| 17 | <code>for (int k = 0; k &lt; n; k++) acc += A[row * n + k] * B[k * n + col];</code> | Dot product of row of A and column of B. Matrices are stored row by row: element (r, c) is at r × n + c. |
| 18 | <code>C[row * n + col] = acc;</code> | Write the result once. |
| 19 | <code>}</code> | End of the if. |
| 20 | <code>}</code> | End of the kernel. |
| 22 | <code>__global__ void mmTiled(const float* A, const float* B, float* C, int n) {</code> | The tiled kernel. |
| 23 | <code>__shared__ float sA[TILE][TILE];</code> | Shared tile of A, one per block. |
| 24 | <code>__shared__ float sB[TILE][TILE];</code> | Shared tile of B. |
| 25 | <code>int tx = threadIdx.x, ty = threadIdx.y;</code> | Position of the thread inside the tile. |
| 26 | <code>int row = blockIdx.y * TILE + ty, col = blockIdx.x * TILE + tx;</code> | Output position. The same as in the naive kernel. |
| 27 | <code>float acc = 0.0f;</code> | Accumulator. |
| 28 | <code>for (int p = 0; p &lt; (n + TILE - 1) / TILE; p++) {</code> | One phase per tile along the shared dimension. Ceil covers a partial last tile. |
| 29 | <code>int ka = p * TILE + tx, kb = p * TILE + ty;</code> | Column of A and row of B this thread loads in this phase. |
| 30 | <code>sA[ty][tx] = (row &lt; n &amp;&amp; ka &lt; n) ? A[row * n + ka] : 0.0f;</code> | Load one value of A, or 0 outside the matrix. |
| 31 | <code>sB[ty][tx] = (kb &lt; n &amp;&amp; col &lt; n) ? B[kb * n + col] : 0.0f;</code> | Load one value of B, or 0 outside the matrix. |
| 32 | <code>__syncthreads();</code> | Wait until both tiles are fully loaded. |
| 33 | <code>for (int k = 0; k &lt; TILE; k++) acc += sA[ty][k] * sB[k][tx];</code> | TILE multiply adds from shared memory. |
| 34 | <code>__syncthreads();</code> | Wait before the next phase overwrites the tiles. |
| 35 | <code>}</code> | End of the phase loop. |
| 36 | <code>if (row &lt; n &amp;&amp; col &lt; n) C[row * n + col] = acc;</code> | Write the result if this thread owns a real output. |
| 37 | <code>}</code> | End of the kernel. |
| 39 | <code>void mmCPU(const float* A, const float* B, float* C, int n) {</code> | CPU reference, so we know the right answer. |
| 40 | <code>for (int i = 0; i &lt; n; i++)</code> | Every row. |
| 41 | <code>for (int j = 0; j &lt; n; j++) {</code> | Every column. |
| 42 | <code>double acc = 0.0;</code> | Sum in double for an accurate reference. |
| 43 | <code>for (int k = 0; k &lt; n; k++) acc += (double)A[i * n + k] * B[k * n + j];</code> | The dot product on the CPU. |
| 44 | <code>C[i * n + j] = (float)acc;</code> | Store it as float. |
| 45 | <code>}</code> | End of the column loop. |
| 46 | <code>}</code> | End of the function. |
| 48 | <code>int main(int argc, char** argv) {</code> | main can take the size from the command line, for example ./matmul 512. |
| 49 | <code>int n = argc &gt; 1 ? atoi(argv[1]) : 1024;</code> | Size n, 1024 by default. |
| 50 | <code>size_t bytes = (size_t)n * n * sizeof(float);</code> | Bytes per matrix. The cast to size_t avoids overflow for large n. |
| 51 | <code>float *A = (float*)malloc(bytes), *B = (float*)malloc(bytes), *C = (float*)malloc(bytes), *R = (float*)malloc(bytes);</code> | Host matrices: two inputs, the GPU result and the CPU reference. |
| 52 | <code>for (int i = 0; i &lt; n * n; i++) { A[i] = (rand() % 100) / 100.0f; B[i] = (rand() % 100) / 100.0f; }</code> | Fill the inputs with values 0.00 to 0.99. |
| 53 | <code>mmCPU(A, B, R, n);</code> | Compute the reference once. For n = 1024 this takes a few seconds on the CPU. |
| 55 | <code>float *dA, *dB, *dC;</code> | Device pointers. |
| 56 | <code>CUDA_CHECK(cudaMalloc(&amp;dA, bytes));</code> | GPU memory for A. |
| 57 | <code>CUDA_CHECK(cudaMalloc(&amp;dB, bytes));</code> | GPU memory for B. |
| 58 | <code>CUDA_CHECK(cudaMalloc(&amp;dC, bytes));</code> | GPU memory for C. |
| 59 | <code>CUDA_CHECK(cudaMemcpy(dA, A, bytes, cudaMemcpyHostToDevice));</code> | Copy A to the GPU. |
| 60 | <code>CUDA_CHECK(cudaMemcpy(dB, B, bytes, cudaMemcpyHostToDevice));</code> | Copy B to the GPU. |
| 62 | <code>dim3 block(TILE, TILE), grid((n + TILE - 1) / TILE, (n + TILE - 1) / TILE);</code> | TILE × TILE threads per block and enough blocks to cover the n × n output. |
| 63 | <code>cudaEvent_t start, stop;</code> | Two events to time the kernels on the GPU. |
| 64 | <code>CUDA_CHECK(cudaEventCreate(&amp;start));</code> | Create the start event. |
| 65 | <code>CUDA_CHECK(cudaEventCreate(&amp;stop));</code> | Create the stop event. |
| 67 | <code>for (int v = 0; v &lt; 2; v++) {</code> | v = 0 runs the naive kernel, v = 1 the tiled one. |
| 68 | <code>CUDA_CHECK(cudaMemset(dC, 0, bytes));</code> | Clear C, so a kernel that writes nothing cannot pass the check by accident. |
| 69 | <code>CUDA_CHECK(cudaEventRecord(start));</code> | Time stamp before the kernel. |
| 70 | <code>if (v == 0) mmNaive&lt;&lt;&lt;grid, block&gt;&gt;&gt;(dA, dB, dC, n);</code> | Launch the naive kernel. |
| 71 | <code>else mmTiled&lt;&lt;&lt;grid, block&gt;&gt;&gt;(dA, dB, dC, n);</code> | Or the tiled kernel. |
| 72 | <code>CUDA_CHECK(cudaGetLastError());</code> | Catch launch errors such as a bad block size. |
| 73 | <code>CUDA_CHECK(cudaEventRecord(stop));</code> | Time stamp after the kernel. |
| 74 | <code>CUDA_CHECK(cudaEventSynchronize(stop));</code> | Wait until the GPU reaches the stop event. |
| 75 | <code>float ms = 0.0f;</code> | A float for the time. |
| 76 | <code>CUDA_CHECK(cudaEventElapsedTime(&amp;ms, start, stop));</code> | Time between the events in milliseconds. |
| 77 | <code>CUDA_CHECK(cudaMemcpy(C, dC, bytes, cudaMemcpyDeviceToHost));</code> | Copy the result back. |
| 78 | <code>double maxErr = 0.0;</code> | Largest difference from the CPU answer. |
| 79 | <code>for (int i = 0; i &lt; n * n; i++) maxErr = fmax(maxErr, fabs(C[i] - R[i]));</code> | Compare every element. |
| 80 | <code>double gflops = 2.0 * n * n * n / (ms * 1e6);</code> | 2n³ FLOPs ÷ (ms × 1e6) = GFLOP/s. |
| 81 | <code>printf(&quot;%-6s n=%d  %.3f ms  %.1f GFLOP/s  max error %.2e\n&quot;, v == 0 ? &quot;naive&quot; : &quot;tiled&quot;, n, ms, gflops, maxErr);</code> | One line per kernel with its time, speed and error. An error around 1e-4 is normal float rounding. |
| 82 | <code>}</code> | End of the loop. |
| 84 | <code>cudaFree(dA); cudaFree(dB); cudaFree(dC);</code> | Free GPU memory. |
| 85 | <code>free(A); free(B); free(C); free(R);</code> | Free CPU memory. |
| 86 | <code>return 0;</code> | Success. |
| 87 | <code>}</code> | End of main. |

In [ ]:
!nvcc -O3 -arch=sm_75 -lineinfo -o matmul matmul.cu
!./matmul 1024

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>!nvcc -O3 -arch=sm_75 -lineinfo -o matmul matmul.cu</code> | Compile for the T4 (sm_75). -lineinfo keeps source line numbers so the tools below can point at lines. |
| 2 | <code>!./matmul 1024</code> | Run with n = 1024. The tiled kernel should be clearly faster than the naive one. |

## 2. compute-sanitizer: find memory bugs

compute-sanitizer runs the program and checks every memory access. It is slow, so we use a small size.
Try it again after removing a bounds check from the kernels.

In [ ]:
!compute-sanitizer --tool memcheck ./matmul 100

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>!compute-sanitizer --tool memcheck ./matmul 100</code> | memcheck reports out of bounds and misaligned accesses with the kernel, thread and source line. n = 100 is not a multiple of 16, so the bounds checks are tested. |

## 3. Nsight Systems: the timeline

nsys records every copy and kernel on a timeline. --stats=true prints a summary table, so we do not need the GUI.

In [ ]:
!nsys profile --stats=true -o matmul_report ./matmul 1024

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>!nsys profile --stats=true -o matmul_report ./matmul 1024</code> | Profiles the whole run. Look at the CUDA GPU Kernel Summary table for the time of mmNaive and mmTiled, and the memory operations table for the copies. |

## 4. Nsight Compute: inside one kernel

ncu measures one kernel in detail: achieved occupancy, memory throughput, and how busy the SMs are.
On some cloud machines it needs extra permissions and prints ERR_NVGPUCTRPERM. Then run it on your own machine instead.

In [ ]:
!ncu --kernel-name mmTiled --launch-count 1 --section SpeedOfLight --section Occupancy ./matmul 1024

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>!ncu --kernel-name mmTiled --launch-count 1 --section SpeedOfLight --section Occupancy ./matmul 1024</code> | Profile only the first launch of mmTiled. SpeedOfLight shows compute and memory use as a percent of peak, Occupancy shows active warps against the limit. |

## 5. Try these

1. Change TILE to 32 and recompile. Compare the time and the occupancy that ncu reports.
2. Pad sA to [TILE][TILE + 1]. Does it change the time? Use chapter 3.3 to explain.
3. Make each thread compute 2 outputs in the same column (a 2 × 1 register tile). Measure the GFLOP/s again.